# Training a LeNet-style classifier on CIFAR-10

Look at a photograph at 32 × 32 pixels and you can often still tell a ship
from a truck. Can a small network learn to do the same? We will adapt the LeNet
architecture to color photographs, train it for four epochs, and see what it
gets right and where it makes mistakes.

Use a standard CPU runtime in Colab, or a local Jupyter kernel with PyTorch,
TorchVision, and Matplotlib installed. The first run downloads the CIFAR-10
archive (about 163 MiB). TorchVision reuses it on subsequent runs.

## Loading the training and validation images

CIFAR-10 has 50,000 training images and 10,000 test images. We will run a short
exercise with 12,000 of the training images. We will use another 2,000 to check
how the model is doing after each epoch. This is our **validation set**. We
reserve the official test images for the final evaluation.

TorchVision supplies the dataset and the image transforms. We convert each
image to a tensor and scale each color channel from the range [0, 1] to [−1, 1].

We download the original archive from MIT Open Learning's public mirror. TorchVision checks it against the original CIFAR-10 checksum.

In [ ]:
from copy import deepcopy
from pathlib import Path
from time import perf_counter

import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

seed = 7
batch_size = 128
torch.manual_seed(seed)
torch.set_num_threads(min(4, torch.get_num_threads()))

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])
class CIFAR10(datasets.CIFAR10):
    url = (
        "https://huggingface.co/datasets/MIT-OL-AI-D/cifar-10-python/"
        "resolve/a48007227f9e2cd0af96175f4afb5ac4965e261b/cifar-10-python.tar.gz"
    )


dataset_root = Path("data") / "lenet"
training_images = CIFAR10(
    dataset_root, train=True, download=True, transform=transform,
)
training_set, validation_set, _ = random_split(
    training_images, [12_000, 2_000, 36_000],
    generator=torch.Generator().manual_seed(seed),
)
train_loader = DataLoader(
    training_set, batch_size=batch_size, shuffle=True,
    generator=torch.Generator().manual_seed(seed),
)
validation_loader = DataLoader(validation_set, batch_size=batch_size)
class_names = training_images.classes
print(f"Training: {len(training_set):,}   Validation: {len(validation_set):,}")

Here are six images from the training set. The labels tell us which answer the
model should learn to give. We undo the normalization only when displaying the
images so their colors look the way they did before preprocessing.

In [ ]:
output_dir = Path("outputs") / "lenet"
output_dir.mkdir(parents=True, exist_ok=True)

fig, axes = plt.subplots(1, 6, figsize=(8.5, 1.95), layout="constrained")
fig.suptitle("Examples from the CIFAR-10 training set", fontweight="bold")
for index, ax in enumerate(axes):
    image, label = training_set[index]
    ax.imshow((image / 2 + 0.5).permute(1, 2, 0))
    ax.set_title(class_names[label])
    ax.set_axis_off()
fig.savefig(output_dir / "cifar10_training_examples.png", dpi=300)
plt.show()

## Building the network

LeNet was designed for grayscale images. Our CIFAR-10 images have red, green,
and blue channels, so the first convolution, `cn1`, takes three channels.
Each convolution is followed by a ReLU activation and max pooling. Pooling
keeps the largest value in each 2 × 2 window and halves the grid size here.

The spatial grid goes from 32 × 32 to 28 × 28 after `cn1`, then to 14 × 14
after pooling. The second convolution and pooling reduce it to 5 × 5. We
flatten the 16 channels of that final grid so the fully connected layers can
produce ten scores, one per class.

In [ ]:
class LeNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.cn1 = nn.Conv2d(3, 6, kernel_size=5)
        self.cn2 = nn.Conv2d(6, 16, kernel_size=5)
        self.fc1 = nn.Linear(16 * 5 * 5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    def forward(self, images):
        features = F.max_pool2d(F.relu(self.cn1(images)), 2)
        features = F.max_pool2d(F.relu(self.cn2(features)), 2)
        features = torch.flatten(features, start_dim=1)
        features = F.relu(self.fc1(features))
        features = F.relu(self.fc2(features))
        return self.fc3(features)


lenet = LeNet()
image, _ = training_set[0]
with torch.inference_mode():
    scores = lenet(image.unsqueeze(0))
assert scores.shape == (1, len(class_names))
print("One image produces", scores.shape[1], "class scores.")

## Training and choosing a checkpoint

One epoch is one pass through the training subset. We feed the model a batch
at a time, compute cross-entropy loss from its class scores, and let Adam
update its weights. A validation pass uses the same loss calculation, with the
weights held fixed.

We multiply each batch's mean loss by its image count before adding it to the
total. That way each image has the same weight when we calculate the epoch's
mean, even if the last batch is smaller.

In [ ]:
criterion = nn.CrossEntropyLoss()


def run_epoch(model, loader, optimizer=None):
    training = optimizer is not None
    model.train(training)
    loss_total = 0.0
    correct = 0
    examples = 0

    with torch.set_grad_enabled(training):
        for images, labels in loader:
            if training:
                optimizer.zero_grad()
            scores = model(images)
            loss = criterion(scores, labels)
            if training:
                loss.backward()
                optimizer.step()

            count = labels.size(0)
            loss_total += loss.item() * count
            correct += (scores.argmax(dim=1) == labels).sum().item()
            examples += count

    return loss_total / examples, correct / examples

In [ ]:
optimizer = torch.optim.Adam(lenet.parameters(), lr=0.001)
best_validation_loss = float("inf")
best_epoch = None
best_weights = None

started = perf_counter()
for epoch in range(1, 5):
    training_loss, _ = run_epoch(lenet, train_loader, optimizer)
    validation_loss, validation_accuracy = run_epoch(lenet, validation_loader)
    print(
        f"Epoch {epoch} | train loss {training_loss:.3f} | "
        f"validation loss {validation_loss:.3f} | "
        f"validation accuracy {validation_accuracy:.1%}"
    )
    if validation_loss < best_validation_loss:
        best_validation_loss = validation_loss
        best_epoch = epoch
        best_weights = deepcopy(lenet.state_dict())

lenet.load_state_dict(best_weights)
print(f"Selected epoch {best_epoch} by validation loss.")
print(f"Training and validation took {perf_counter() - started:.1f} seconds.")

## Checking predictions on the test set

We have used validation loss to choose the weights. Now we can see how those
weights perform on all 10,000 test images. With ten equally common classes,
random guesses would be correct about 10% of the time.

The figure below shows the first six test examples alongside their actual
labels and the model's predictions. Do you see any pairs that look similar at
this size? The test accuracy tells us how often the model was right across
the whole set. The six examples let us see a few of those decisions.

In [ ]:
test_set = datasets.CIFAR10(dataset_root, train=False, transform=transform)
test_loader = DataLoader(test_set, batch_size=batch_size)
test_loss, test_accuracy = run_epoch(lenet, test_loader)
print(f"Test loss {test_loss:.3f} | accuracy {test_accuracy:.1%} "
      f"on {len(test_set):,} images")

images, labels = next(iter(test_loader))
with torch.inference_mode():
    predictions = lenet(images[:6]).argmax(dim=1)

fig, axes = plt.subplots(1, 6, figsize=(8.5, 2.3), layout="constrained")
fig.suptitle("LeNet predictions on six test images", fontweight="bold")
for image, label, prediction, ax in zip(images[:6], labels[:6], predictions, axes):
    ax.imshow((image / 2 + 0.5).permute(1, 2, 0))
    ax.set_title(f"Actual: {class_names[label]}\nPredicted: {class_names[prediction]}",
                 fontsize=9)
    ax.set_axis_off()
fig.savefig(output_dir / "lenet_test_predictions.png", dpi=300)
plt.show()

In this exercise we trained a small network from scratch and saw how its
convolutions and pooling layers turn a photograph into class scores. As we
move to larger CNNs, we will see the same operations arranged in different
ways, including blocks where the input follows more than one path.

[Original CIFAR-10 dataset and technical report](https://cave.cs.toronto.edu/kriz/cifar.html)

[MIT Open Learning archive mirror](https://huggingface.co/datasets/MIT-OL-AI-D/cifar-10-python)